# Notebook 1: simulación y reconstrucción de una Single Pixel Camera

**Proyecto INF449: SPC + 3D Gaussian Splatting.** Este notebook cubre dos pasos del pipeline:

1. **Simular** las mediciones de una Single Pixel Camera sobre cada vista de una escena.
2. **Reconstruir** las vistas desde esas mediciones y medir su calidad contra la imagen original.

Las vistas reconstruidas conservan el nombre de su pose COLMAP, de modo que el
[Notebook 2](02_gaussian_splatting.ipynb) pueda entrenar 3DGS con ellas. Las decisiones de diseño
están argumentadas en `docs/00_plan_y_decisiones.md`.

El notebook es re-ejecutable: cada vista y cada configuración terminada se guarda en disco y se omite
al volver a correr.

## 1. Configuración

Todos los parámetros están en la celda siguiente. `papermill` puede sobrescribirlos por línea de
comandos (celda con la etiqueta `parameters`).

In [ ]:
# Datos
DATASET = "tandt/truck"       # relativo a data/tandt_db: tandt/truck, tandt/train, db/drjohnson, db/playroom
NUM_VIEWS = None              # None = todas; un entero toma vistas equiespaciadas
RUN_NAME = "completa"         # subcarpeta en outputs/spc_runs/<escena>/

# Rejilla de medición: potencias de 2 por el requisito de Hadamard, con H×W ≈ píxeles de la vista
GRID_H = 512
GRID_W = 1024

# Experimento principal
RATIOS = [0.05, 0.10, 0.25, 0.50]          # M = ratio * N, por canal
PATTERNS = ["hadamard_cc", "random"]       # hadamard_cc, hadamard_rd, random, fourier
METHODS = ["direct", "tv"]
SEED = 449

# Modelo del sensor (y está normalizada: 1 = todos los espejos encendidos sobre una escena blanca)
PHOTONS = 1e9                 # fotones por patrón a fondo de escala; None = sin ruido de disparo
READ_NOISE = 1e-5             # desviación del ruido gaussiano de lectura; 0 = sin él
ADC_BITS = 16                 # cuantización; None = sin cuantizar
DC_REPEATS = 16               # repeticiones del patrón "todo encendido" (referencia de los coeficientes)

# Reconstrucción TV-FISTA
TV_ITERS = 100
TV_INNER = 8
TV_TOL = 1e-4
LAMBDA_GRID = [1e-3, 3e-3, 1e-2, 3e-2]
LAMBDA_VIEWS = 4              # vistas de entrenamiento usadas para elegir lambda

# Estudios complementarios en pocas vistas (órdenes, Bernoulli, ruido, color)
RUN_PILOT_STUDIES = True
PILOT_VIEWS = 3
PILOT_LAMBDAS = [3e-3, 1e-2, 3e-2]

JPEG_QUALITY = 100            # los nombres de pose terminan en .jpg
LLFF_HOLD = 8                 # el repo de 3DGS usa 1 de cada 8 vistas como test

## 2. Entorno

El notebook se detiene con un mensaje claro si no hay CUDA: `.venv` debe tener `torch` con CUDA.

In [ ]:
import csv
import json
import os
import shutil
import sys
import time
import zlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from PIL import Image


def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / "third_party" / "gaussian-splatting").is_dir() and (path / "data").is_dir():
            return path
    raise FileNotFoundError(f"No encontré la raíz del repositorio desde {start}.")


ROOT = find_repo_root(Path.cwd().resolve())
GS_REPO = ROOT / "third_party" / "gaussian-splatting"
sys.path.insert(0, str(GS_REPO))

print("Python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA de torch", torch.version.cuda)
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No hay CUDA en este entorno. Este notebook debe correr en la GPU: selecciona el kernel de "
        ".venv, que tiene torch 2.11.0+cu128."
    )
DEVICE = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | VRAM: {props.total_memory / 2**30:.2f} GiB")

torch.manual_seed(SEED)
np.random.seed(SEED)

from lpipsPyTorch.modules.lpips import LPIPS
from scene.colmap_loader import read_extrinsics_binary
from utils.image_utils import psnr as gs_psnr
from utils.loss_utils import ssim as gs_ssim

SCENE_DIR = ROOT / "data" / "tandt_db" / DATASET
SCENE_NAME = Path(DATASET).name
RUN_DIR = ROOT / "outputs" / "spc_runs" / SCENE_NAME / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
N_PIXELS = GRID_H * GRID_W
print("Escena:", SCENE_DIR.relative_to(ROOT), "| Resultados:", RUN_DIR.relative_to(ROOT))


class OOMGuard:
    """Convierte un OOM de CUDA en un mensaje accionable."""

    def __init__(self, label):
        self.label = label

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc_type is not None and issubclass(exc_type, torch.cuda.OutOfMemoryError):
            raise RuntimeError(
                f"Sin VRAM en '{self.label}'. Reserva actual: {torch.cuda.memory_reserved() / 2**30:.2f} GiB. "
                "Cierra otros procesos que usen la GPU; si persiste, baja GRID_H/GRID_W."
            ) from exc
        return False

In [ ]:
# Marca de inicio para medir el tiempo total de esta ejecución (sección 10)
RUN_T0 = time.time()

## 3. Vistas y poses

Los nombres de vista se leen de `images.bin`, no de la carpeta. Así se evita el error de los intentos
previos: renombrar a `01.jpg`, `02.jpg`… rompía la correspondencia con las poses. El orden y el split
train/test son los del lector del repo: orden por nombre y 1 de cada 8 vistas para test.

In [ ]:
extrinsics = read_extrinsics_binary(str(SCENE_DIR / "sparse" / "0" / "images.bin"))
pose_names = sorted(Path(e.name).name for e in extrinsics.values())
files = {p.name for p in (SCENE_DIR / "images").iterdir()}
missing = sorted(set(pose_names) - files)
if missing:
    raise RuntimeError(f"{len(missing)} poses sin imagen, por ejemplo {missing[:3]}.")

if NUM_VIEWS is None or NUM_VIEWS >= len(pose_names):
    VIEWS = pose_names
else:
    VIEWS = [pose_names[i] for i in np.unique(np.linspace(0, len(pose_names) - 1, NUM_VIEWS).astype(int))]
SPLIT = {name: ("test" if i % LLFF_HOLD == 0 else "train") for i, name in enumerate(VIEWS)}
TRAIN_VIEWS = [v for v in VIEWS if SPLIT[v] == "train"]
TEST_VIEWS = [v for v in VIEWS if SPLIT[v] == "test"]


def load_rgb(path):
    with Image.open(path) as image:
        array = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1).contiguous().to(DEVICE)


def evenly(items, count):
    return [items[i] for i in np.unique(np.linspace(0, len(items) - 1, count).astype(int))]


IMG_H, IMG_W = load_rgb(SCENE_DIR / "images" / VIEWS[0]).shape[-2:]
print(f"Vistas: {len(VIEWS)} ({len(TRAIN_VIEWS)} train, {len(TEST_VIEWS)} test) | archivo {IMG_W}x{IMG_H} | "
      f"rejilla {GRID_W}x{GRID_H} (N = {N_PIXELS})")

## 4. Operadores de medición

Una SPC mide productos internos entre la escena y máscaras del DMD: $y = \Phi x + \text{ruido}$, con
$M = \text{ratio}\cdot N$ filas. A esta resolución $\Phi$ tendría $524\,288^2$ entradas si fuera
completa, así que nunca se almacena: se aplica con transformadas rápidas en la GPU.

| Patrón | Máscaras | Selección de las $M$ filas |
|---|---|---|
| `hadamard_cc` | Walsh-Hadamard, binarias | *Cake-cutting*: menos bloques conexos primero |
| `hadamard_rd` | Walsh-Hadamard, binarias | *Russian dolls*: subrejillas anidadas, y cake-cutting dentro de cada nivel |
| `random` | Hadamard con píxeles permutados, binarias y pseudoaleatorias | Azar, con una semilla distinta por vista |
| `fourier` | Sinusoides de 4 fases, con grises (un DMD real necesitaría *dithering*) | Baja frecuencia radial |

**Orden cake-cutting analítico.** Un patrón separable $w_i(y)\,w_j(x)$ con secuencias $s_i, s_j$ tiene
exactamente $(s_i+1)(s_j+1)$ bloques conexos, así que no hace falta etiquetar 524 288 imágenes.

**Aleatorio estructurado.** Una matriz Bernoulli i.i.d. no cabe a esta resolución (1.7 GB a 5 % y 17 GB a
50 %, a 1 bit por entrada). Se usa una matriz estructuralmente aleatoria (Do et al., 2012):
$\Phi = S\,H\,\Pi$. La sección 9.2 la compara con Bernoulli densa a baja resolución.

Los tres operadores son ortonormales por filas ($\Phi\Phi^\top = I$), lo que fija el paso de FISTA en 1.

In [ ]:
def fwht_last(x):
    """Transformada de Walsh-Hadamard ortonormal, en orden natural (Sylvester), sobre el último eje."""
    n = x.shape[-1]
    levels = int(np.log2(n))
    if 2**levels != n:
        raise ValueError(f"Hadamard exige una potencia de 2; recibió {n}.")
    shape = x.shape[:-1]
    h = 1
    for _ in range(levels):
        y = x.reshape(*shape, n // (2 * h), 2, h)
        a, b = y[..., 0, :], y[..., 1, :]
        x = torch.stack((a + b, a - b), dim=-2).reshape(*shape, n)
        h *= 2
    return x / np.sqrt(n)


def wht2(x):
    """Transformada 2D separable; es su propia inversa."""
    return fwht_last(fwht_last(x).transpose(-1, -2)).transpose(-1, -2)


def sequency(n):
    """Número de cambios de signo de cada fila de la matriz de Hadamard natural."""
    idx = np.arange(n)
    parity = np.array([[bin(i & j).count("1") % 2 for j in idx] for i in idx], dtype=np.int8)
    return (parity[:, 1:] != parity[:, :-1]).sum(axis=1)


_ORDER_CACHE = {}


def hadamard_order(height, width, kind):
    """Índices aplanados de las filas 2D, de más a menos prioritaria."""
    key = (height, width, kind)
    if key in _ORDER_CACHE:
        return _ORDER_CACHE[key]
    sy, sx = np.meshgrid(sequency(height), sequency(width), indexing="ij")
    blocks = (sy + 1) * (sx + 1)
    flat = np.arange(height * width)
    if kind == "cc":
        order = np.lexsort((flat, blocks.ravel()))
    elif kind == "rd":
        # Nivel L: el patrón es constante en la subrejilla H/2^(n-L) x W/2^(m-L), respetando el aspecto.
        extra_y = max(0, int(np.log2(width / height)))
        extra_x = max(0, int(np.log2(height / width)))
        level = np.maximum(np.ceil(np.log2(sy + 1)) + extra_y, np.ceil(np.log2(sx + 1)) + extra_x)
        order = np.lexsort((flat, blocks.ravel(), level.ravel().astype(int)))
    else:
        raise ValueError(f"Orden desconocido: {kind}")
    _ORDER_CACHE[key] = order
    return order


class SelectedOp:
    """Filas seleccionadas de una base ortonormal. La fila 0 siempre es el patrón DC."""

    differential = False  # True si cada coeficiente usa un par de máscaras complementarias

    @property
    def n_coeffs(self):
        raise NotImplementedError

    @property
    def n_patterns(self):
        """Mediciones físicas por canal: el DC cuenta aparte con DC_REPEATS."""
        count = self.n_coeffs - 1
        return (2 * count if self.differential else count) + DC_REPEATS


class HadamardOp(SelectedOp):
    def __init__(self, height, width, m, order_kind):
        self.height, self.width = height, width
        self.mask_scale = 1.0 / np.sqrt(height * width)
        self.idx = torch.from_numpy(hadamard_order(height, width, order_kind)[:m].copy()).to(DEVICE)

    @property
    def n_coeffs(self):
        return self.idx.numel()

    def forward(self, x):
        return wht2(x).flatten(-2)[..., self.idx]

    def adjoint(self, c):
        z = torch.zeros(*c.shape[:-1], self.height * self.width, device=c.device, dtype=c.dtype)
        z[..., self.idx] = c
        return wht2(z.reshape(*c.shape[:-1], self.height, self.width))


class ScrambledHadamardOp(SelectedOp):
    """Phi = S H Pi. Cada fila es una máscara binaria pseudoaleatoria, regenerable desde la semilla."""

    def __init__(self, height, width, m, seed):
        self.height, self.width = height, width
        self.mask_scale = 1.0 / np.sqrt(height * width)
        n = height * width
        generator = torch.Generator().manual_seed(seed)
        self.perm = torch.randperm(n, generator=generator).to(DEVICE)
        self.inv = torch.argsort(self.perm)
        rows = torch.randperm(n - 1, generator=generator)[: m - 1].to(DEVICE) + 1
        self.idx = torch.cat((torch.zeros(1, dtype=torch.long, device=DEVICE), rows))

    @property
    def n_coeffs(self):
        return self.idx.numel()

    def forward(self, x):
        scrambled = x.flatten(-2)[..., self.perm].reshape(x.shape)
        return wht2(scrambled).flatten(-2)[..., self.idx]

    def adjoint(self, c):
        z = torch.zeros(*c.shape[:-1], self.height * self.width, device=c.device, dtype=c.dtype)
        z[..., self.idx] = c
        scrambled = wht2(z.reshape(*c.shape[:-1], self.height, self.width))
        return scrambled.flatten(-2)[..., self.inv].reshape(scrambled.shape)


class FourierOp(SelectedOp):
    """Coeficientes reales ortonormales de la FFT. Cada uno usa dos fases complementarias (4 por complejo)."""

    differential = True

    def __init__(self, height, width, m):
        self.height, self.width = height, width
        self.mask_scale = np.sqrt(2.0 / (height * width))
        fy = torch.fft.fftfreq(height, device=DEVICE)
        fx = torch.fft.fftfreq(width, device=DEVICE)
        gy, gx = torch.meshgrid(fy, fx, indexing="ij")
        half = (gx > 0) | ((gx == 0) & (gy > 0))
        half &= (gx != -0.5) & (gy != -0.5)  # Nyquist: autoconjugados, innecesarios a estos ratios
        candidates = torch.nonzero(half.flatten()).squeeze(1)
        radius = (gy.flatten()[candidates] ** 2 + gx.flatten()[candidates] ** 2)
        chosen = candidates[torch.argsort(radius, stable=True)[: max(1, (m - 1) // 4)]]
        self.iy, self.ix = chosen // width, chosen % width
        self.iyc, self.ixc = (-self.iy) % height, (-self.ix) % width

    @property
    def n_coeffs(self):
        return 1 + 2 * self.iy.numel()

    def forward(self, x):
        spectrum = torch.fft.fft2(x, norm="ortho")
        values = spectrum[..., self.iy, self.ix] * np.sqrt(2.0)
        return torch.cat((spectrum[..., 0, 0].real[..., None], values.real, values.imag), dim=-1)

    def adjoint(self, c):
        k = self.iy.numel()
        spectrum = torch.zeros(*c.shape[:-1], self.height, self.width, device=c.device, dtype=torch.complex64)
        values = torch.complex(c[..., 1:k + 1], c[..., k + 1:]) / np.sqrt(2.0)
        spectrum[..., 0, 0] = c[..., 0].to(torch.complex64)
        spectrum[..., self.iy, self.ix] = values
        spectrum[..., self.iyc, self.ixc] = values.conj()
        return torch.fft.ifft2(spectrum, norm="ortho").real


class BayerOp:
    """Máscara de color RGGB delante del operador de un canal: 3 canales en una sola serie de mediciones."""

    def __init__(self, base):
        self.base = base
        mask = torch.zeros(3, base.height, base.width, device=DEVICE)
        mask[0, 0::2, 0::2] = 1
        mask[1, 0::2, 1::2] = 1
        mask[1, 1::2, 0::2] = 1
        mask[2, 1::2, 1::2] = 1
        self.mask = mask

    def mosaic(self, x):
        return (x * self.mask).sum(dim=-3, keepdim=True)

    def forward(self, x):
        return self.base.forward(self.mosaic(x))

    def adjoint(self, c):
        return self.base.adjoint(c) * self.mask

    def demosaic(self, mono):
        """Interpolación bilineal por convolución normalizada, para el método directo."""
        kernel = torch.tensor([[1.0, 2.0, 1.0], [2.0, 4.0, 2.0], [1.0, 2.0, 1.0]], device=DEVICE)[None, None]
        kernel = kernel.repeat(3, 1, 1, 1)
        num = F.conv2d((mono * self.mask)[None], kernel, padding=1, groups=3)
        den = F.conv2d(self.mask[None], kernel, padding=1, groups=3)
        return (num / den)[0]


def view_seed(view_name):
    return (SEED + zlib.crc32(view_name.encode("utf-8"))) % (2**31)


def make_op(pattern, ratio, view_name, height=None, width=None):
    height, width = height or GRID_H, width or GRID_W
    m = max(2, int(round(ratio * height * width)))
    if pattern == "hadamard_cc":
        return HadamardOp(height, width, m, "cc")
    if pattern == "hadamard_rd":
        return HadamardOp(height, width, m, "rd")
    if pattern == "random":
        return ScrambledHadamardOp(height, width, m, view_seed(view_name))
    if pattern == "fourier":
        return FourierOp(height, width, m)
    raise ValueError(f"Patrón desconocido: {pattern}")

### Pruebas de los operadores

Estas comprobaciones corren en cada ejecución sobre una rejilla pequeña y detienen el notebook si algo
falla. Verifican propiedades matemáticas, no solo que el código corra.

In [ ]:
def dense_matrix(op, height, width):
    n = height * width
    basis = torch.eye(n, device=DEVICE).reshape(n, 1, height, width)
    return torch.cat([op.forward(basis[i:i + 256]) for i in range(0, n, 256)], dim=0)[:, 0].T


def check_operators():
    h, w = 16, 32
    x = torch.rand(3, h, w, device=DEVICE)
    for name, op in [
        ("hadamard_cc", make_op("hadamard_cc", 0.25, "t", h, w)),
        ("hadamard_rd", make_op("hadamard_rd", 0.25, "t", h, w)),
        ("random", make_op("random", 0.25, "t", h, w)),
        ("fourier", make_op("fourier", 0.25, "t", h, w)),
    ]:
        c = torch.rand(3, op.n_coeffs, device=DEVICE)
        lhs, rhs = (op.forward(x) * c).sum().item(), (x * op.adjoint(c)).sum().item()
        assert abs(lhs - rhs) < 1e-2, f"{name}: el adjunto no coincide ({lhs} vs {rhs})"
        phi = dense_matrix(op, h, w)
        identity_error = (phi @ phi.T - torch.eye(phi.shape[0], device=DEVICE)).abs().max().item()
        assert identity_error < 1e-4, f"{name}: Phi Phi^T != I (error {identity_error})"
        dc_error = abs(op.forward(x)[0, 0].item() - x[0].mean().item() * np.sqrt(h * w))
        assert dc_error < 1e-3, f"{name}: la fila 0 no es el patrón DC"

    # Un operador completo es invertible
    full = make_op("random", 1.0, "t", h, w)
    assert (full.adjoint(full.forward(x)) - x).abs().max() < 1e-4, "random al 100 % no reconstruye"

    # Las máscaras físicas (1 + h)/2 son binarias
    phi = dense_matrix(make_op("random", 0.25, "t", h, w), h, w)
    masks = (1 + phi * np.sqrt(h * w)) / 2
    assert ((masks - masks.round()).abs().max() < 1e-4), "random: las máscaras no son binarias"
    assert abs(masks[1:].mean().item() - 0.5) < 0.02, "random: las máscaras no encienden ~50 % de los espejos"

    # Cake-cutting: comparación contra los bloques contados por fuerza bruta
    from scipy.ndimage import label
    phi_cc = dense_matrix(make_op("hadamard_cc", 1.0, "t", h, w), h, w)
    patterns = (phi_cc > 0).reshape(-1, h, w).cpu().numpy()
    for i in range(0, h * w, 37):
        regions = label(patterns[i])[1] + label(~patterns[i])[1]
        idx = hadamard_order(h, w, "cc")[i]
        sy, sx = sequency(h)[idx // w], sequency(w)[idx % w]
        assert regions == (sy + 1) * (sx + 1), f"cake-cutting: {regions} regiones en el patrón {i}"

    # Russian dolls: al 25 % equivale a promediar por bloques de 2x2
    rd = make_op("hadamard_rd", 0.25, "t", h, w)
    pooled = F.avg_pool2d(x[None], 2)[0].repeat_interleave(2, -1).repeat_interleave(2, -2)
    assert (rd.adjoint(rd.forward(x)) - pooled).abs().max() < 1e-4, "russian dolls: no reproduce el promedio 2x2"

    # Bayer: el mosaico tiene una cuarta parte de R y B, y el adjunto es consistente
    bayer = BayerOp(make_op("random", 0.5, "t", h, w))
    c = torch.rand(1, bayer.base.n_coeffs, device=DEVICE)
    lhs, rhs = (bayer.forward(x) * c).sum().item(), (x * bayer.adjoint(c)).sum().item()
    assert abs(lhs - rhs) < 1e-2, "Bayer: el adjunto no coincide"
    assert bayer.mask[0].mean() == 0.25 and bayer.mask[1].mean() == 0.5, "Bayer: proporciones incorrectas"
    print("Operadores OK: adjunto, Phi Phi^T = I, máscaras binarias, conteo cake-cutting, Russian dolls y Bayer.")


check_operators()

## 5. Paso 1: simulación de las mediciones

**Modelo físico.** Un DMD solo enciende o apaga espejos, de modo que un patrón ±1 no se puede mostrar
directamente. Se simula la máscara $P = (1 + h)/2$ y la medición normalizada

$$y_k = \frac{1}{N}\sum_i P_k(i)\,x_i = \frac{1}{2}\left(\bar x + \frac{c_k}{\sqrt N}\right),$$

donde $c_k$ es el coeficiente ortonormal y $\bar x$ el promedio, medido con el patrón "todo encendido".
El coeficiente se recupera con $\hat c_k = \sqrt N\,(2\hat y_k - \hat{\bar x})$. **Esto amplifica el ruido
del sensor por $2\sqrt N$**: la multiplexación de un solo píxel exige un rango dinámico alto a esta
resolución. La sección 9.3 lo cuantifica.

El sensor aplica, en orden: ruido de disparo (Poisson, aproximado por una gaussiana porque hay más de
$10^5$ fotones), ruido gaussiano de lectura y cuantización a `ADC_BITS`.

**Resolución.** La vista de 979×546 se remuestrea a la rejilla 1024×512, y al final se vuelve al tamaño
del archivo. El cambio de aspecto equivale a macro-píxeles no cuadrados en el DMD. La condición de
control `control_resample` mide cuánto cuesta este ida y vuelta por sí solo.

**Color.** El experimento principal mide cada canal por separado (3M mediciones). Es lo que permite
comparar el ratio directamente con la literatura, y además corresponde a una rueda de color o a tres
detectores con separadores dicroicos. La sección 9.4 lo compara con una máscara Bayer.

In [ ]:
def to_grid(x):
    return F.interpolate(x[None], size=(GRID_H, GRID_W), mode="bicubic", antialias=True, align_corners=False)[0].clamp(0, 1)


def to_file_size(x):
    return F.interpolate(x[None], size=(IMG_H, IMG_W), mode="bicubic", antialias=True, align_corners=False)[0].clamp(0, 1)


def sensor(y, generator, photons=None, read_noise=None, bits=None):
    """Aplica el modelo de sensor y devuelve códigos (o valores flotantes si no hay ADC)."""
    photons = PHOTONS if photons == "default" else photons
    out = y
    if photons:
        out = out + torch.sqrt(y.clamp(min=0) / photons) * torch.randn(y.shape, device=y.device, generator=generator)
    if read_noise:
        out = out + read_noise * torch.randn(y.shape, device=y.device, generator=generator)
    if bits:
        return (out.clamp(0, 1) * (2**bits - 1)).round()
    return out


def noise_generator(*key):
    generator = torch.Generator(device=DEVICE)
    generator.manual_seed((SEED + zlib.crc32("|".join(map(str, key)).encode("utf-8"))) % (2**31))
    return generator


def measure(x, op, generator, photons, read_noise, bits):
    """Simula las mediciones físicas de x [C, H, W]. Devuelve los códigos del sensor y del DC."""
    dc = x.mean(dim=(-2, -1))[:, None]
    ac = op.forward(x)[..., 1:] * op.mask_scale
    y = torch.cat((0.5 * (dc + ac), 0.5 * (dc - ac)), dim=-1) if op.differential else 0.5 * (dc + ac)
    codes = sensor(y, generator, photons, read_noise, bits)
    dc_codes = sensor(dc.repeat(1, DC_REPEATS), generator, photons, read_noise, bits)
    return codes, dc_codes


def decode(codes, dc_codes, op, bits):
    """Invierte el modelo físico: de códigos del sensor a coeficientes ortonormales."""
    scale = (2**bits - 1) if bits else 1.0
    y = codes / scale
    dc = (dc_codes / scale).mean(dim=-1, keepdim=True)
    if op.differential:
        half = y.shape[-1] // 2
        ac = (y[..., :half] - y[..., half:]) / op.mask_scale
    else:
        ac = (2.0 * y - dc) / op.mask_scale
    return torch.cat((dc * np.sqrt(op.height * op.width), ac), dim=-1)


def check_measure_roundtrip():
    x = torch.rand(3, 16, 32, device=DEVICE)
    for pattern in ["hadamard_cc", "random", "fourier"]:
        op = make_op(pattern, 0.5, "t", 16, 32)
        codes, dc_codes = measure(x, op, noise_generator("t"), None, 0, None)
        error = (decode(codes, dc_codes, op, None) - op.forward(x)).abs().max().item()
        assert error < 1e-3, f"{pattern}: medir y decodificar no devuelve los coeficientes (error {error})"
    print("Modelo físico OK: sin ruido, medir y decodificar devuelve los coeficientes.")


check_measure_roundtrip()

In [ ]:
def config_name(pattern, ratio):
    percent = ratio * 100
    if abs(percent - round(percent)) > 1e-9:
        raise ValueError(f"RATIOS debe ser un porcentaje entero para nombrar la carpeta: {ratio}")
    return f"{pattern}_r{round(percent):03d}"


def save_npz_atomic(path, **arrays):
    tmp = path.with_name(path.name + ".tmp.npz")
    np.savez(tmp, **arrays)
    os.replace(tmp, path)


def code_dtype():
    if ADC_BITS is None:
        return np.float32
    return np.uint16 if ADC_BITS <= 16 else np.uint32


def simulate_config(pattern, ratio):
    """Paso 1. Las máscaras no se guardan: se regeneran desde pattern, ratio y la semilla de cada vista."""
    name = config_name(pattern, ratio)
    out = RUN_DIR / name / "measurements"
    out.mkdir(parents=True, exist_ok=True)
    pending = [v for v in VIEWS if not (out / f"{Path(v).stem}.npz").exists()]
    start = time.perf_counter()
    with OOMGuard(f"simular {name}"):
        for view in pending:
            x = to_grid(load_rgb(SCENE_DIR / "images" / view))
            op = make_op(pattern, ratio, view)
            codes, dc_codes = measure(x, op, noise_generator(pattern, ratio, view), PHOTONS, READ_NOISE, ADC_BITS)
            save_npz_atomic(out / f"{Path(view).stem}.npz",
                            codes=codes.cpu().numpy().astype(code_dtype()),
                            dc_codes=dc_codes.cpu().numpy().astype(code_dtype()))
    op = make_op(pattern, ratio, VIEWS[0])
    config = {
        "pattern": pattern, "ratio": ratio, "grid": [GRID_H, GRID_W], "seed": SEED,
        "view_seed": "(SEED + crc32(nombre_vista)) % 2^31",
        "photons": PHOTONS, "read_noise": READ_NOISE, "adc_bits": ADC_BITS, "dc_repeats": DC_REPEATS,
        "n_coeffs_per_channel": op.n_coeffs, "n_patterns_per_channel": op.n_patterns,
        "n_measurements_total": 3 * op.n_patterns, "color": "por canal",
        "views": len(VIEWS),
    }
    (RUN_DIR / name / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    return {"config": name, "simuladas": len(pending), "ya_existian": len(VIEWS) - len(pending),
            "tiempo_s": round(time.perf_counter() - start, 1), "patrones_por_canal": op.n_patterns}


def load_measurements(pattern, ratio, view):
    path = RUN_DIR / config_name(pattern, ratio) / "measurements" / f"{Path(view).stem}.npz"
    with np.load(path) as data:
        codes = torch.from_numpy(data["codes"].astype(np.float32)).to(DEVICE)
        dc_codes = torch.from_numpy(data["dc_codes"].astype(np.float32)).to(DEVICE)
    return codes, dc_codes


rows = [simulate_config(pattern, ratio) for pattern in PATTERNS for ratio in RATIOS]
display(pd.DataFrame(rows))
disk_mb = sum(f.stat().st_size for f in RUN_DIR.rglob("*.npz")) / 2**20
print(f"Mediciones en disco: {disk_mb:.0f} MB")

## 6. Paso 2: reconstrucción

**(a) Directa.** $\hat x = \Phi^\top \hat c$, con ceros en los coeficientes no medidos. Como
$\Phi\Phi^\top = I$, es la solución de mínima norma. No usa ningún prior.

**(b) TV-FISTA.** Resuelve

$$\min_{0 \le x \le 1}\ \tfrac12\|\Phi x - \hat c\|_2^2 + \lambda\,\mathrm{TV}(x)$$

con FISTA (Beck y Teboulle, 2009), paso 1 porque $\|\Phi\|=1$ y el operador de proximidad resuelto con FGP
y *warm start* del dual. La variación total es conjunta entre canales, lo que reduce los halos de color.

**Elección de $\lambda$.** Se busca por patrón y ratio en vistas de **entrenamiento**, nunca de test, para
que la evaluación no quede sesgada. Solo entonces se reconstruyen todas las vistas.

In [ ]:
def tv_prox(v, lam, p, q, n_inner):
    """Prox de TV isotrópica conjunta con caja [0, 1], por FGP (Beck y Teboulle, 2009). Dual con warm start."""
    r, s = p, q
    t = 1.0
    step = 1.0 / (8.0 * lam)
    for _ in range(n_inner):
        div = r + s
        div[..., 1:] -= r[..., :-1]
        div[..., 1:, :] -= s[..., :-1, :]
        x = (v - lam * div).clamp(0, 1)
        pn, qn = r.clone(), s.clone()
        pn[..., :-1] += step * (x[..., :-1] - x[..., 1:])
        qn[..., :-1, :] += step * (x[..., :-1, :] - x[..., 1:, :])
        scale = torch.sqrt((pn * pn + qn * qn).sum(dim=-3, keepdim=True)).clamp(min=1.0)
        pn, qn = pn / scale, qn / scale
        t_new = (1.0 + np.sqrt(1.0 + 4.0 * t * t)) / 2.0
        beta = (t - 1.0) / t_new
        r, s = pn + beta * (pn - p), qn + beta * (qn - q)
        p, q, t = pn, qn, t_new
    div = p + q
    div[..., 1:] -= p[..., :-1]
    div[..., 1:, :] -= q[..., :-1, :]
    return (v - lam * div).clamp(0, 1), p, q


@torch.no_grad()
def tv_fista(op, c, x0, lam, n_iter=None, n_inner=None, tol=None, step=1.0):
    n_iter, n_inner, tol = n_iter or TV_ITERS, n_inner or TV_INNER, TV_TOL if tol is None else tol
    x = x0.clamp(0, 1)
    z = x.clone()
    p, q = torch.zeros_like(x), torch.zeros_like(x)
    t = 1.0
    iterations = 0
    for iterations in range(1, n_iter + 1):
        v = z - step * op.adjoint(op.forward(z) - c)
        x_new, p, q = tv_prox(v, lam * step, p, q, n_inner)
        t_new = (1.0 + np.sqrt(1.0 + 4.0 * t * t)) / 2.0
        z = x_new + ((t - 1.0) / t_new) * (x_new - x)
        change = (x_new - x).norm() / x.norm().clamp(min=1e-8)
        x, t = x_new, t_new
        if change < tol:
            break
    return x, iterations


def check_tv():
    """TV debe reducir el error frente a la solución directa, con pocas mediciones en una imagen suave a trozos."""
    h, w = 64, 128
    yy, xx = torch.meshgrid(torch.arange(h, device=DEVICE), torch.arange(w, device=DEVICE), indexing="ij")
    x = ((xx > 40) & (yy > 20)).float()[None].repeat(3, 1, 1) * 0.7 + 0.1
    op = make_op("random", 0.25, "t", h, w)
    direct = op.adjoint(op.forward(x))
    recon, _ = tv_fista(op, op.forward(x), direct, 1e-3, n_iter=150, tol=0)
    e_direct, e_tv = (direct - x).norm().item(), (recon - x).norm().item()
    assert e_tv < 0.5 * e_direct, f"TV no mejora lo esperado: {e_tv:.3f} vs directa {e_direct:.3f}"
    print(f"TV-FISTA OK: error {e_tv:.3f} frente a {e_direct:.3f} de la solución directa.")


check_tv()

lpips_net = LPIPS("vgg").to(DEVICE).eval()


@torch.no_grad()
def image_metrics(image, reference):
    """Mismas funciones que 3DGS en el paper: PSNR, SSIM y LPIPS-VGG del repo."""
    a, b = image[None], reference[None]
    return {"psnr": gs_psnr(a, b).item(), "ssim": gs_ssim(a, b).item(), "lpips": lpips_net(a, b).item()}

In [ ]:
LAMBDA_FILE = RUN_DIR / "lambda_search.csv"


def search_lambdas():
    """Elige lambda por (patrón, ratio) usando solo vistas de entrenamiento. Reanuda desde el CSV."""
    views = evenly(TRAIN_VIEWS, LAMBDA_VIEWS)
    done = pd.read_csv(LAMBDA_FILE) if LAMBDA_FILE.exists() else pd.DataFrame(columns=["pattern", "ratio", "lam", "view", "psnr"])
    seen = set(zip(done["pattern"], done["ratio"].round(6), done["lam"].round(9), done["view"]))
    is_new = not LAMBDA_FILE.exists()
    with open(LAMBDA_FILE, "a", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        if is_new:
            writer.writerow(["pattern", "ratio", "lam", "view", "psnr"])
        with OOMGuard("búsqueda de lambda"):
            for pattern in PATTERNS:
                for ratio in RATIOS:
                    for view in views:
                        gt = load_rgb(SCENE_DIR / "images" / view)
                        op, cache = make_op(pattern, ratio, view), None
                        for lam in LAMBDA_GRID:
                            if (pattern, round(ratio, 6), round(lam, 9), view) in seen:
                                continue
                            if cache is None:
                                codes, dc_codes = load_measurements(pattern, ratio, view)
                                cache = decode(codes, dc_codes, op, ADC_BITS)
                            recon, _ = tv_fista(op, cache, op.adjoint(cache), lam)
                            writer.writerow([pattern, ratio, lam, view, gs_psnr(to_file_size(recon)[None], gt[None]).item()])
                            handle.flush()
    table = pd.read_csv(LAMBDA_FILE)
    table = table[table["pattern"].isin(PATTERNS) & table["ratio"].isin(RATIOS) & table["lam"].isin(LAMBDA_GRID)]
    return table.groupby(["pattern", "ratio", "lam"], as_index=False)["psnr"].mean()


start = time.perf_counter()
lambda_table = search_lambdas() if "tv" in METHODS else pd.DataFrame()
print(f"Búsqueda de lambda: {time.perf_counter() - start:.0f} s, en {LAMBDA_VIEWS} vistas de entrenamiento")
if len(lambda_table):
    display(lambda_table.pivot_table(index=["pattern", "ratio"], columns="lam", values="psnr").round(2))
    best = lambda_table.loc[lambda_table.groupby(["pattern", "ratio"])["psnr"].idxmax()]
    BEST_LAMBDA = {(row.pattern, round(row.ratio, 6)): row.lam for row in best.itertuples()}
    on_edge = [k for k, v in BEST_LAMBDA.items() if v in (min(LAMBDA_GRID), max(LAMBDA_GRID))]
    if on_edge:
        print("ATENCIÓN: lambda óptimo en el borde de la rejilla para", on_edge)

In [ ]:
FIELDS = ["view", "split", "psnr", "ssim", "lpips", "seconds", "iterations"]


def save_jpeg_atomic(image, path):
    array = (image.permute(1, 2, 0).cpu().numpy() * 255.0).round().astype(np.uint8)
    tmp = path.with_name(path.name + ".tmp")
    Image.fromarray(array).save(tmp, format="JPEG", quality=JPEG_QUALITY, subsampling=0)
    os.replace(tmp, path)


def reconstruct_condition(out_dir, produce):
    """Recorre las vistas pendientes, guarda la imagen con el nombre de su pose y la mide desde el archivo."""
    images_dir = out_dir / "images"
    images_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = out_dir / "metrics_per_view.csv"
    done = set()
    if metrics_path.exists():
        done = {v for v in pd.read_csv(metrics_path)["view"] if (images_dir / v).exists()}
        kept = pd.read_csv(metrics_path)
        kept[kept["view"].isin(done)].to_csv(metrics_path, index=False)
    is_new = not metrics_path.exists()
    with open(metrics_path, "a", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=FIELDS)
        if is_new:
            writer.writeheader()
        for view in VIEWS:
            if view in done:
                continue
            gt = load_rgb(SCENE_DIR / "images" / view)
            torch.cuda.synchronize()
            start = time.perf_counter()
            image, iterations = produce(view, gt)
            torch.cuda.synchronize()
            seconds = time.perf_counter() - start
            save_jpeg_atomic(image, images_dir / view)
            metrics = image_metrics(load_rgb(images_dir / view), gt)
            writer.writerow({"view": view, "split": SPLIT[view], **metrics, "seconds": seconds, "iterations": iterations})
            handle.flush()
    table = pd.read_csv(metrics_path)
    missing = sorted(set(VIEWS) - set(table["view"]))
    if missing:
        raise RuntimeError(f"{out_dir}: faltan {len(missing)} vistas en las métricas.")
    return table[table["view"].isin(VIEWS)]


def produce_direct(pattern, ratio):
    def produce(view, gt):
        op = make_op(pattern, ratio, view)
        codes, dc_codes = load_measurements(pattern, ratio, view)
        return to_file_size(op.adjoint(decode(codes, dc_codes, op, ADC_BITS)).clamp(0, 1)), 0
    return produce


def produce_tv(pattern, ratio, lam):
    def produce(view, gt):
        op = make_op(pattern, ratio, view)
        codes, dc_codes = load_measurements(pattern, ratio, view)
        c = decode(codes, dc_codes, op, ADC_BITS)
        recon, iterations = tv_fista(op, c, op.adjoint(c), lam)
        return to_file_size(recon), iterations
    return produce


def produce_control(view, gt):
    return to_file_size(to_grid(gt)), 0


entries, summaries = [], []


def register(table, **info):
    name = info["name"]
    images = RUN_DIR / info["relative"] / "images"
    entries.append({**info, "images_dir": str(images.relative_to(ROOT)).replace(os.sep, "/")})
    for split, part in [("todas", table), ("test", table[table["split"] == "test"])]:
        summaries.append({
            "condicion": name, "pattern": info["pattern"], "ratio": info["ratio"], "method": info["method"],
            "lam": info.get("lam"), "split": split, "vistas": len(part),
            "psnr": part["psnr"].mean(), "psnr_std": part["psnr"].std(),
            "ssim": part["ssim"].mean(), "ssim_std": part["ssim"].std(),
            "lpips": part["lpips"].mean(), "lpips_std": part["lpips"].std(),
            "s_por_vista": part["seconds"].mean(), "iter_media": part["iterations"].mean(),
        })


with OOMGuard("control de remuestreo"):
    table = reconstruct_condition(RUN_DIR / "control_resample", produce_control)
register(table, name="control_resample", relative="control_resample", pattern="control", ratio=1.0, method="control")
print(f"control_resample: PSNR {table['psnr'].mean():.2f} dB (cota por el remuestreo y el JPEG)")

for pattern in PATTERNS:
    for ratio in RATIOS:
        base = config_name(pattern, ratio)
        config = json.loads((RUN_DIR / base / "config.json").read_text(encoding="utf-8"))
        for method in METHODS:
            start = time.perf_counter()
            lam = BEST_LAMBDA[(pattern, round(ratio, 6))] if method == "tv" else None
            producer = produce_tv(pattern, ratio, lam) if method == "tv" else produce_direct(pattern, ratio)
            with OOMGuard(f"reconstruir {base}/{method}"):
                table = reconstruct_condition(RUN_DIR / base / method, producer)
            register(table, name=f"{base}_{method}", relative=f"{base}/{method}", pattern=pattern, ratio=ratio,
                     method=method, lam=lam, n_measurements_total=config["n_measurements_total"])
            print(f"{base}/{method}: PSNR {table['psnr'].mean():.2f} SSIM {table['ssim'].mean():.3f} "
                  f"LPIPS {table['lpips'].mean():.3f} | {time.perf_counter() - start:.0f} s, "
                  f"{table['seconds'].mean():.2f} s/vista", flush=True)
            torch.cuda.empty_cache()

summary_2d = pd.DataFrame(summaries)
summary_2d.to_csv(RUN_DIR / "summary_2d.csv", index=False)

manifest = {
    "dataset": DATASET, "scene": SCENE_NAME, "run": RUN_NAME, "views": VIEWS, "file_size": [IMG_H, IMG_W],
    "grid": [GRID_H, GRID_W], "seed": SEED, "sensor": {"photons": PHOTONS, "read_noise": READ_NOISE, "adc_bits": ADC_BITS},
    "jpeg_quality": JPEG_QUALITY, "conditions": entries,
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Manifiesto para el Notebook 2:", (RUN_DIR / "manifest.json").relative_to(ROOT))

## 7. Resultados por ratio y método

Métricas contra la imagen original, siempre a la resolución del archivo y leyendo el JPEG guardado, que
es exactamente lo que recibe 3DGS.

In [ ]:
view_table = summary_2d[summary_2d["split"] == "todas"]
columns = ["condicion", "ratio", "psnr", "psnr_std", "ssim", "lpips", "s_por_vista", "iter_media"]
display(view_table[columns].round(3).sort_values(["ratio", "condicion"]).reset_index(drop=True))

In [ ]:
# Paleta categórica validada (guía dataviz): el color sigue a la serie, y cada serie lleva etiqueta directa.
SERIES = {
    ("hadamard_cc", "tv"): ("Hadamard + TV", "#2a78d6"),
    ("hadamard_cc", "direct"): ("Hadamard directa", "#eb6834"),
    ("random", "tv"): ("Aleatorio + TV", "#1baf7a"),
    ("random", "direct"): ("Aleatorio directa", "#eda100"),
}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": GRID, "axes.labelcolor": MUTED,
    "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK, "axes.grid": True, "grid.color": GRID,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "axes.titlesize": 11,
    "figure.dpi": 110, "savefig.dpi": 160, "savefig.bbox": "tight",
})

METRICS = [("psnr", "PSNR (dB), mayor es mejor"), ("ssim", "SSIM, mayor es mejor"), ("lpips", "LPIPS-VGG, menor es mejor")]
control = view_table[view_table["method"] == "control"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for axis, (metric, title) in zip(axes, METRICS):
    for (pattern, method), (label, color) in SERIES.items():
        part = view_table[(view_table["pattern"] == pattern) & (view_table["method"] == method)].sort_values("ratio")
        if part.empty:
            continue
        axis.plot(part["ratio"] * 100, part[metric], color=color, lw=2, marker="o", ms=6, label=label)
        axis.fill_between(part["ratio"] * 100, part[metric] - part[f"{metric}_std"], part[metric] + part[f"{metric}_std"],
                          color=color, alpha=0.12, lw=0)
    if len(control):
        axis.axhline(control[metric].iloc[0], color=MUTED, lw=1, ls="--", label="Control (solo remuestreo)")
    axis.set_title(title, loc="left")
    axis.set_xlabel("Ratio de muestreo M/N (%)")
    axis.set_xticks([r * 100 for r in RATIOS])
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(labels), frameon=False, fontsize=9)
fig.suptitle(f"Calidad 2D de las vistas reconstruidas, {SCENE_NAME}: media y ±1 desviación entre vistas", x=0.01, ha="left")
fig.tight_layout(rect=(0, 0.07, 1, 0.95))
fig.savefig(RUN_DIR / "fig_calidad_vs_ratio.png")
plt.show()

In [ ]:
# Lado a lado: una vista de test. Filas: condiciones; columnas: GT, reconstrucción y error ampliado.
view = TEST_VIEWS[len(TEST_VIEWS) // 2]
gt = load_rgb(SCENE_DIR / "images" / view)
examples = [(s["pattern"], s["ratio"], s["method"]) for s in entries if s["method"] in METHODS and s["pattern"] == PATTERNS[0] and s["method"] == "tv"]
examples += [(s["pattern"], 0.10 if 0.10 in RATIOS else RATIOS[0], s["method"]) for s in entries
             if s["pattern"] in PATTERNS and s["ratio"] == (0.10 if 0.10 in RATIOS else RATIOS[0]) and (s["pattern"], s["method"]) != (PATTERNS[0], "tv")]
fig, axes = plt.subplots(len(examples), 3, figsize=(13, 2.5 * len(examples)), squeeze=False)
for row, (pattern, ratio, method) in enumerate(examples):
    path = RUN_DIR / config_name(pattern, ratio) / method / "images" / view
    recon = load_rgb(path)
    metrics = image_metrics(recon, gt)
    axes[row, 0].imshow(gt.permute(1, 2, 0).cpu())
    axes[row, 0].set_title(f"Original {view}" if row == 0 else "", fontsize=10)
    axes[row, 1].imshow(recon.permute(1, 2, 0).cpu())
    axes[row, 1].set_title(f"{pattern} / {method}, {ratio:.0%}: {metrics['psnr']:.1f} dB, SSIM {metrics['ssim']:.3f}", fontsize=10)
    axes[row, 2].imshow(((recon - gt).abs() * 4).clamp(0, 1).mean(0).cpu(), cmap="magma", vmin=0, vmax=1)
    axes[row, 2].set_title("|error| x 4" if row == 0 else "", fontsize=10)
    for axis in axes[row]:
        axis.axis("off")
fig.tight_layout()
fig.savefig(RUN_DIR / "fig_gt_recon_error.png")
plt.show()

## 8. Comprobaciones de salida

Antes de pasar al Notebook 2 se valida el contrato: cada condición tiene todas las vistas, con su nombre
de pose exacto, el tamaño del archivo original y JPEG legible.

In [ ]:
for entry in entries:
    folder = ROOT / entry["images_dir"]
    names = sorted(p.name for p in folder.iterdir())
    assert names == sorted(VIEWS), f"{entry['name']}: los nombres no coinciden con las poses"
    with Image.open(folder / VIEWS[0]) as image:
        assert image.size == (IMG_W, IMG_H), f"{entry['name']}: tamaño {image.size}"
leftovers = list(RUN_DIR.rglob("*.tmp")) + list(RUN_DIR.rglob("*.tmp.npz"))
assert not leftovers, f"Quedaron archivos temporales: {leftovers[:3]}"
print(f"Contrato OK: {len(entries)} condiciones con {len(VIEWS)} vistas cada una.")

## 9. Estudios complementarios

Estos estudios usan pocas vistas de entrenamiento y sirven para justificar decisiones, no para entrenar
3DGS. Como eligen $\lambda$ con la métrica de esas mismas vistas, sus cifras son **comparaciones internas
optimistas**: no se deben mezclar con las tablas de la sección 7.

Cada resultado se guarda en un CSV y no se recalcula al volver a correr.

In [ ]:
PILOT = evenly(TRAIN_VIEWS, PILOT_VIEWS) if RUN_PILOT_STUDIES else []


def pilot_csv(name, compute):
    path = RUN_DIR / f"estudio_{name}.csv"
    if path.exists():
        return pd.read_csv(path)
    start = time.perf_counter()
    with OOMGuard(f"estudio {name}"):
        table = pd.DataFrame(compute())
    table.to_csv(path, index=False)
    print(f"Estudio {name}: {time.perf_counter() - start:.0f} s")
    return table


def best_tv(op, c, gt, x0=None, finish=None):
    finish = finish or to_file_size
    x0 = op.adjoint(c) if x0 is None else x0
    best = None
    for lam in PILOT_LAMBDAS:
        recon, _ = tv_fista(op, c, x0, lam)
        image = finish(recon)
        metrics = image_metrics(image, gt)
        if best is None or metrics["psnr"] > best["psnr"]:
            best = {**metrics, "lam": lam}
    return best


def mean_rows(rows, keys):
    return pd.DataFrame(rows).groupby(keys, as_index=False, sort=False).mean(numeric_only=True).to_dict("records")

### 9.1 Órdenes de patrón y Fourier

Mismo ruido y misma rejilla. El ratio cuenta mediciones físicas, así que Fourier obtiene la mitad de
coeficientes reales que Hadamard (dos fases complementarias por coeficiente).

In [ ]:
def study_patterns():
    rows = []
    for view in PILOT:
        gt = load_rgb(SCENE_DIR / "images" / view)
        x = to_grid(gt)
        for pattern in ["hadamard_cc", "hadamard_rd", "random", "fourier"]:
            for ratio in RATIOS:
                op = make_op(pattern, ratio, view)
                codes, dc_codes = measure(x, op, noise_generator("patrones", pattern, ratio, view), PHOTONS, READ_NOISE, ADC_BITS)
                c = decode(codes, dc_codes, op, ADC_BITS)
                base = {"pattern": pattern, "ratio": ratio}
                rows.append({**base, "method": "direct", **image_metrics(to_file_size(op.adjoint(c).clamp(0, 1)), gt)})
                rows.append({**base, "method": "tv", **best_tv(op, c, gt)})
    return mean_rows(rows, ["pattern", "ratio", "method"])


if RUN_PILOT_STUDIES:
    patterns_table = pilot_csv("patrones", study_patterns)
    display(patterns_table.pivot_table(index=["pattern", "method"], columns="ratio", values="psnr", sort=False).round(2))
    display(patterns_table.pivot_table(index=["pattern", "method"], columns="ratio", values="lpips", sort=False).round(3))

### 9.2 ¿La matriz estructurada se comporta como Bernoulli?

A 512×1024 una matriz Bernoulli densa no cabe, pero a 64×128 sí. Sin ruido, se comparan en esa rejilla el
operador `random` ($S H \Pi$) y Bernoulli ±1 i.i.d. normalizada. Como Bernoulli no es ortonormal por filas,
FISTA usa el paso $1/L$, con $L$ estimado por iteración de potencia.

In [ ]:
class DenseOp:
    def __init__(self, phi):
        self.phi = phi

    def forward(self, x):
        return x.flatten(-2) @ self.phi.T

    def adjoint(self, c):
        return (c @ self.phi).reshape(*c.shape[:-1], 64, 128)


def study_bernoulli():
    rows = []
    n = 64 * 128
    for view in PILOT:
        gt = load_rgb(SCENE_DIR / "images" / view)
        x = F.interpolate(gt[None], size=(64, 128), mode="bicubic", antialias=True, align_corners=False)[0].clamp(0, 1)
        for ratio in RATIOS:
            m = int(round(ratio * n))
            srm = make_op("random", ratio, view, 64, 128)
            generator = torch.Generator(device=DEVICE).manual_seed(view_seed(view))
            phi = (2.0 * torch.randint(0, 2, (m, n), device=DEVICE, generator=generator).float() - 1.0) / np.sqrt(n)
            dense = DenseOp(phi)
            v = torch.randn(n, device=DEVICE)
            for _ in range(30):
                v = phi.T @ (phi @ v)
                lipschitz = v.norm().item()
                v = v / v.norm()
            for name, op, step in [("estructurada", srm, 1.0), ("bernoulli_densa", dense, 1.0 / lipschitz)]:
                c = op.forward(x)
                best = None
                for lam in PILOT_LAMBDAS:
                    recon, _ = tv_fista(op, c, (step * op.adjoint(c)).clamp(0, 1), lam, n_iter=300, tol=0, step=step)
                    metric = gs_psnr(recon[None], x[None]).item()
                    best = max(best or metric, metric)
                rows.append({"matriz": name, "ratio": ratio, "psnr": best})
            del phi, dense
    torch.cuda.empty_cache()
    return mean_rows(rows, ["matriz", "ratio"])


if RUN_PILOT_STUDIES:
    bernoulli_table = pilot_csv("bernoulli", study_bernoulli)
    display(bernoulli_table.pivot_table(index="matriz", columns="ratio", values="psnr", sort=False).round(2))

### 9.3 Sensibilidad al ruido y a la linealidad de la luz

A ratio 25 % se varía el número de fotones por patrón. `None` y 0 significan sensor ideal, sin ADC.

Además se prueba medir en **luz lineal**. Las imágenes del dataset están codificadas en sRGB, pero un
fotodiodo integra radiancia lineal. La simulación principal mide los valores sRGB, igual que casi toda la
literatura de simulación SPC: esta fila indica cuánto cambia el resultado si se linealiza primero.

In [ ]:
def srgb_to_linear(x):
    return torch.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055) ** 2.4)


def linear_to_srgb(x):
    x = x.clamp(0, 1)
    return torch.where(x <= 0.0031308, x * 12.92, 1.055 * x ** (1 / 2.4) - 0.055)


def study_noise():
    rows = []
    ratio = 0.25
    settings = [("ideal", None, 0, None, False), ("1e10", 1e10, READ_NOISE, ADC_BITS, False),
                ("1e9 (por defecto)", 1e9, READ_NOISE, ADC_BITS, False), ("1e8", 1e8, READ_NOISE, ADC_BITS, False),
                ("1e7", 1e7, READ_NOISE, ADC_BITS, False), ("1e9, luz lineal", 1e9, READ_NOISE, ADC_BITS, True)]
    for view in PILOT:
        gt = load_rgb(SCENE_DIR / "images" / view)
        for pattern in ["hadamard_cc", "random"]:
            op = make_op(pattern, ratio, view)
            for name, photons, read_noise, bits, linear in settings:
                x = to_grid(gt)
                x = srgb_to_linear(x) if linear else x
                finish = (lambda r: to_file_size(linear_to_srgb(r))) if linear else to_file_size
                codes, dc_codes = measure(x, op, noise_generator("ruido", name, pattern, view), photons, read_noise, bits)
                c = decode(codes, dc_codes, op, bits)
                base = {"sensor": name, "pattern": pattern}
                rows.append({**base, "method": "direct", **image_metrics(finish(op.adjoint(c).clamp(0, 1)), gt)})
                rows.append({**base, "method": "tv", **best_tv(op, c, gt, finish=finish)})
    return mean_rows(rows, ["sensor", "pattern", "method"])


if RUN_PILOT_STUDIES:
    noise_table = pilot_csv("ruido", study_noise)
    display(noise_table.pivot_table(index="sensor", columns=["pattern", "method"], values="psnr", sort=False).round(2))

### 9.4 Estrategia de color: por canal frente a máscara Bayer

Se compara con **el mismo número total de mediciones**. Una SPC por canal con ratio $r$ gasta $3rN$
mediciones; una máscara Bayer usa todas esas mediciones sobre el mosaico, en una sola serie temporal.

La hipótesis a contrastar: con un orden de baja frecuencia, la crominancia del mosaico queda en altas
secuencias, de modo que Hadamard ordenado debería perder color, mientras que los patrones aleatorios la
conservan.

In [ ]:
def study_color():
    rows = []
    for view in PILOT:
        gt = load_rgb(SCENE_DIR / "images" / view)
        x = to_grid(gt)
        for pattern in ["hadamard_cc", "random"]:
            for per_channel in [r for r in RATIOS if r * 3 <= 1.0]:
                total = per_channel * 3
                # Por canal
                op = make_op(pattern, per_channel, view)
                codes, dc_codes = measure(x, op, noise_generator("color", "c", pattern, per_channel, view), PHOTONS, READ_NOISE, ADC_BITS)
                c = decode(codes, dc_codes, op, ADC_BITS)
                base = {"total_mediciones_sobre_N": round(total, 3), "pattern": pattern, "color": "por canal"}
                rows.append({**base, "method": "direct", **image_metrics(to_file_size(op.adjoint(c).clamp(0, 1)), gt)})
                rows.append({**base, "method": "tv", **best_tv(op, c, gt)})
                # Bayer: una serie de mediciones sobre el mosaico
                bayer = BayerOp(make_op(pattern, total, view))
                mono = bayer.mosaic(x)
                codes, dc_codes = measure(mono, bayer.base, noise_generator("color", "b", pattern, per_channel, view), PHOTONS, READ_NOISE, ADC_BITS)
                c = decode(codes, dc_codes, bayer.base, ADC_BITS)
                direct = bayer.demosaic(bayer.base.adjoint(c)).clamp(0, 1)
                base = {"total_mediciones_sobre_N": round(total, 3), "pattern": pattern, "color": "Bayer"}
                rows.append({**base, "method": "direct", **image_metrics(to_file_size(direct), gt)})
                rows.append({**base, "method": "tv", **best_tv(bayer, c, gt, x0=direct)})
    return mean_rows(rows, ["total_mediciones_sobre_N", "pattern", "color", "method"])


if RUN_PILOT_STUDIES:
    color_table = pilot_csv("color", study_color)
    display(color_table.pivot_table(index=["pattern", "color", "method"], columns="total_mediciones_sobre_N", values="psnr", sort=False).round(2))

In [ ]:
print("Listo. Archivos principales en", RUN_DIR.relative_to(ROOT))
for name in ["summary_2d.csv", "manifest.json", "lambda_search.csv", "fig_calidad_vs_ratio.png", "fig_gt_recon_error.png"]:
    print(" -", name, "OK" if (RUN_DIR / name).exists() else "FALTA")
print(f"VRAM pico: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")

## 10. Entorno, dataset y tiempos

Esta sección deja por escrito lo necesario para citar y reproducir la corrida: versiones de las librerías principales,
GPU, características del dataset y tiempos medidos. Guarda `entorno.json` y `tiempos.csv` en la carpeta de la corrida.
Los tiempos de reconstrucción salen de `summary_2d.csv`; el total cuenta solo esta ejecución del notebook
(si se reanudó una corrida, no incluye lo que ya estaba hecho).

Código y documentación: https://github.com/fralfaro/inf449-proyecto

In [ ]:
import importlib.metadata as md
import platform
import subprocess

from scene.colmap_loader import read_intrinsics_binary

LIBS = ["torch", "torchvision", "numpy", "scipy", "pandas", "matplotlib", "scikit-image", "lpips", "Pillow", "papermill"]
versions = {}
for name in LIBS:
    try:
        versions[name] = md.version(name)
    except md.PackageNotFoundError:
        versions[name] = "no instalada"

try:
    driver = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                            capture_output=True, text=True, check=True).stdout.strip().splitlines()[0]
except Exception:
    driver = "n/d"

manifest_run = json.loads((RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
cameras = read_intrinsics_binary(str(SCENE_DIR / "sparse" / "0" / "cameras.bin"))
cam = next(iter(cameras.values()))
first_view = Image.open(SCENE_DIR / "images" / manifest_run["views"][0])

entorno = {
    "fecha": time.strftime("%Y-%m-%d %H:%M:%S"),
    "python": sys.version.split()[0],
    "sistema": platform.platform(),
    "gpu": props.name,
    "vram_gib": round(props.total_memory / 2**30, 2),
    "driver_nvidia": driver,
    "cuda_torch": torch.version.cuda,
    "librerias": versions,
    "dataset": DATASET,
    "vistas_usadas": len(manifest_run["views"]),
    "tamano_archivo_vista": list(first_view.size),
    "cameras_bin": {"modelo": cam.model, "ancho": int(cam.width), "alto": int(cam.height)},
    "rejilla_medicion": [GRID_H, GRID_W],
    "semilla": SEED,
    "ratios": RATIOS,
    "patrones": PATTERNS,
    "metodos": METHODS,
    "fotones": PHOTONS,
    "ruido_lectura": READ_NOISE,
    "bits_adc": ADC_BITS,
    "tv_iteraciones_max": TV_ITERS,
    "lambda_grid": LAMBDA_GRID,
}
(RUN_DIR / "entorno.json").write_text(json.dumps(entorno, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"GPU: {entorno['gpu']} ({entorno['vram_gib']} GiB) | driver {driver} | CUDA de torch {entorno['cuda_torch']}")
print(f"Python {entorno['python']} | {entorno['sistema']}")
display(pd.DataFrame(versions, index=["versión"]).T)
print(f"Dataset {DATASET}: {entorno['vistas_usadas']} vistas | archivo {first_view.size[0]}x{first_view.size[1]} | "
      f"cameras.bin {cam.model} {cam.width}x{cam.height} | rejilla {GRID_H}x{GRID_W}")

timing = summary_2d[summary_2d["split"] == "todas"].copy()
timing["metodo"] = np.where(timing["condicion"].str.endswith("_tv"), "tv",
                            np.where(timing["condicion"].str.endswith("_direct"), "directa", "control"))
timing["segundos_total_condicion"] = timing["s_por_vista"] * timing["vistas"]
tiempos = (timing.groupby("metodo")
           .agg(condiciones=("condicion", "count"), s_por_vista_min=("s_por_vista", "min"),
                s_por_vista_media=("s_por_vista", "mean"), s_por_vista_max=("s_por_vista", "max"),
                segundos_total=("segundos_total_condicion", "sum"))
           .round(3).reset_index())
total_s = time.time() - RUN_T0
tiempos.loc[len(tiempos)] = ["TOTAL de esta ejecución (notebook completo)", np.nan, np.nan, np.nan, np.nan, round(total_s, 1)]
tiempos.to_csv(RUN_DIR / "tiempos.csv", index=False)
display(tiempos)
print(f"Tiempo total de esta ejecución: {total_s / 60:.1f} min")